George Rodriguez, Jose Larios

Dataset : https://archive.ics.uci.edu/dataset/360/air+quality

In [ ]:
# ============================================================
# Multinomial Logistic Regression: Air Quality Classification
# Categorizing conditions into 3 tiers (Good, Moderate, Poor)
# based on CO levels, O3 levels, and Temperature.
#
# Dataset: AirQualityUCI_short.csv
# ============================================================
import numpy as np                              # numerical computing
import pandas as pd                             # data manipulation
from sklearn.linear_model import LogisticRegression  # logistic regression model
from sklearn.model_selection import (
    train_test_split,                           # split data
    cross_val_score                             # k-fold cross-validation
)
from sklearn.preprocessing import (
    StandardScaler,                             # standardize features
    LabelEncoder                                # convert text labels → numbers
)
from sklearn.metrics import (
    accuracy_score,                             # overall accuracy
    classification_report,                      # per-class precision/recall/f1
    confusion_matrix                            # prediction matrix
)

In [ ]:
#import and upload data:
from google.colab import files
uploaded = files.upload()

Saving AirQualityUCI.csv to AirQualityUCI (8).csv


In [ ]:
df = pd.read_csv('AirQualityUCI.csv')
df.isnull().sum()

,0
Date,114
Time,114
CO(GT),114
PT08.S1(CO),114
NMHC(GT),114
C6H6(GT),114
PT08.S2(NMHC),114
NOx(GT),114
PT08.S3(NOx),114
NO2(GT),114


In [ ]:
df = df.drop(['Date','Time','Unnamed: 15', 'Unnamed: 16'], axis=1)
df = df.replace(-200, np.nan)
df = df.dropna()
df

,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH
0,2.6,1360.0,150.0,11.9,1046.0,166.0,1056.0,113.0,1692.0,1268.0,13.6,48.9,0.7578
1,2.0,1292.0,112.0,9.4,955.0,103.0,1174.0,92.0,1559.0,972.0,13.3,47.7,0.7255
2,2.2,1402.0,88.0,9.0,939.0,131.0,1140.0,114.0,1555.0,1074.0,11.9,54.0,0.7502
3,2.2,1376.0,80.0,9.2,948.0,172.0,1092.0,122.0,1584.0,1203.0,11.0,60.0,0.7867
4,1.6,1272.0,51.0,6.5,836.0,131.0,1205.0,116.0,1490.0,1110.0,11.2,59.6,0.7888
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1226,4.4,1449.0,501.0,19.5,1282.0,254.0,625.0,133.0,2100.0,1569.0,19.1,61.1,1.3345
1227,3.1,1363.0,234.0,15.1,1152.0,189.0,684.0,110.0,1951.0,1495.0,18.2,65.4,1.3529
1228,3.0,1371.0,212.0,14.6,1136.0,174.0,689.0,102.0,1927.0,1471.0,18.1,66.1,1.3579
1229,3.1,1406.0,275.0,13.7,1107.0,167.0,718.0,108.0,1872.0,1384.0,17.7,66.9,1.3422


In [ ]:
# ============================================================
# STEP 2: Combine All Segments into a Single DataFrame
# ============================================================
# Concatenate the 3 groups into one dataset, just like you would
# if pulling data from different customer tables in a database.

X = pd.DataFrame({
    'co_concentration': df['PT08.S1(CO)'],
    'o3_concentration': df['PT08.S5(O3)'],
    'temperature':      df['T']
})

n = len(X)

n_good = n // 3                                  # block 1 size
n_moderate = n // 3                              # block 2 size
n_poor = n - n_good - n_moderate                 # block 3 size

# Create text labels using the same list multiplication method as the original code
labels = (['good'] * n_good +
          ['moderate'] * n_moderate +
          ['poor'] * n_poor)

print("=" * 60)
print("AIR QUALITY DATASET OVERVIEW")
print("=" * 60)
print(f"Total valid samples: {len(X)}")
print(f"Features: {list(X.columns)}")
print(f"Tiers: {sorted(set(labels))}")
print(f"\nTier distribution (based on humidity levels):")
for tier in ['good', 'moderate', 'poor']:
    count = labels.count(tier)
    print(f"  {tier}: {count} samples ({count/len(labels)*100:.1f}%)")

# Show summary statistics per segment
x_temp = X.copy()
x_temp['segment'] = labels
print(f"\nMean values by segment:")
print(x_temp.groupby('segment').mean().round(2))

AIR QUALITY DATASET OVERVIEW
Total valid samples: 827
Features: ['co_concentration', 'o3_concentration', 'temperature']
Tiers: ['good', 'moderate', 'poor']

Tier distribution (based on humidity levels):
  good: 275 samples (33.3%)
  moderate: 275 samples (33.3%)
  poor: 277 samples (33.5%)

Mean values by segment:
          co_concentration  o3_concentration  temperature
segment                                                  
good               1267.88           1121.47        13.97
moderate           1153.40            943.72        15.41
poor               1202.39           1072.06        17.42


In [ ]:
# ============================================================
# STEP 3: Encode Text Labels as Numbers
# ============================================================
# Machine learning models need numeric targets, not text.
# LabelEncoder converts: 'Budget'→0, 'Premium'→1, 'Standard'→2
# (alphabetical order by default)

le = LabelEncoder()                             # create encoder
y = le.fit_transform(labels)                    # fit and transform labels

print(f"\nLabel encoding mapping:")
for text_label, num_label in zip(le.classes_, range(len(le.classes_))):
    print(f"  '{text_label}' → {num_label}")


Label encoding mapping:
  'good' → 0
  'moderate' → 1
  'poor' → 2


In [ ]:
# ============================================================
# STEP 4: Split Data into Training (75%) and Testing (25%)
# ============================================================
# stratify=y ensures proportional representation of each segment
# in both training and testing sets.

X_train, X_test, y_train, y_test = train_test_split(
    X, y,                                       # features and encoded labels
    test_size=0.25,                             # 25% for testing
    random_state=99,                            # reproducibility
    stratify=y                                  # maintain segment proportions
)

print("\n" + "=" * 60)
print("TRAIN/TEST SPLIT")
print("=" * 60)
print(f"Training samples: {len(X_train)} ({len(X_train)/len(X)*100:.0f}%)")
print(f"Testing samples:  {len(X_test)} ({len(X_test)/len(X)*100:.0f}%)")



TRAIN/TEST SPLIT
Training samples: 620 (75%)
Testing samples:  207 (25%)


In [ ]:
# ============================================================
# STEP 5: Standardize Features
# ============================================================
# Income is in tens of thousands, spending in hundreds, visits in
# single digits. Without scaling, income would dominate the model.
# StandardScaler makes all features equally important by default.

scaler = StandardScaler()                       # create scaler
X_train_scaled = scaler.fit_transform(X_train)  # fit on train, then transform
X_test_scaled = scaler.transform(X_test)        # only transform test data

print(f"\nScaling example (co_concentration):")
print(f"  Before: mean={X_train['co_concentration'].mean():.0f}, "
      f"std={X_train['co_concentration'].std():.0f}")
print(f"  After:  mean={X_train_scaled[:, 0].mean():.4f}, "
      f"std={X_train_scaled[:, 0].std():.4f}")


Scaling example (co_concentration):
  Before: mean=1210, std=242
  After:  mean=-0.0000, std=1.0000


In [ ]:
# ============================================================
# STEP 6: Cross-Validation (Before Final Training)
# ============================================================
# Cross-validation splits the TRAINING data into k folds.
# The model is trained on k-1 folds and tested on the remaining fold.
# This is repeated k times, giving k accuracy scores.
# This gives a more reliable estimate of model performance than
# a single train/test split.

model = LogisticRegression(
    solver='lbfgs',                             # L-BFGS optimizer
    max_iter=500,                               # allow enough iterations
    random_state=99                             # reproducibility
)
# NOTE: For 3+ classes, sklearn automatically uses multinomial (softmax).

# Perform 5-fold cross-validation on the TRAINING set only
cv_scores = cross_val_score(
    model,                                      # the model to evaluate
    X_train_scaled,                             # scaled training features
    y_train,                                    # training labels
    cv=5,                                       # 5 folds
    scoring='accuracy'                          # metric to use
)

print("\n" + "=" * 60)
print("5-FOLD CROSS-VALIDATION RESULTS")
print("=" * 60)
print(f"Fold accuracies: {[f'{s:.4f}' for s in cv_scores]}")
print(f"Mean accuracy:   {cv_scores.mean():.4f}")
print(f"Std deviation:   {cv_scores.std():.4f}")
print(f"(Low std means the model is consistent across different data splits)")


5-FOLD CROSS-VALIDATION RESULTS
Fold accuracies: ['0.4355', '0.5081', '0.5081', '0.4839', '0.4839']
Mean accuracy:   0.4839
Std deviation:   0.0265
(Low std means the model is consistent across different data splits)


In [ ]:
# ============================================================
# STEP 7: Train Final Model and Evaluate on Test Set
# ============================================================
# Now we train on the FULL training set and evaluate on the
# held-out test set that the model has never seen before.

model.fit(X_train_scaled, y_train)              # train on full training set
y_pred = model.predict(X_test_scaled)           # predict on test set

print("\n" + "=" * 60)
print("FINAL MODEL EVALUATION (Test Set)")
print("=" * 60)

# Overall accuracy
test_accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.1f}%)")

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print(f"\nConfusion Matrix:")
print(f"              Predicted")
print(f"              good  moderate  poor")
for i, name in enumerate(le.classes_):
    print(f"  Actual {name:>8s}: {cm[i]}")

# Classification Report
print(f"\nClassification Report:")
print(classification_report(
    y_test, y_pred,
    target_names=le.classes_                    # use original text labels
))


FINAL MODEL EVALUATION (Test Set)
Test Accuracy: 0.5169 (51.7%)

Confusion Matrix:
              Predicted
              good  moderate  poor
  Actual     good: [43 18  8]
  Actual moderate: [21 27 21]
  Actual     poor: [17 15 37]

Classification Report:
              precision    recall  f1-score   support

        good       0.53      0.62      0.57        69
    moderate       0.45      0.39      0.42        69
        poor       0.56      0.54      0.55        69

    accuracy                           0.52       207
   macro avg       0.51      0.52      0.51       207
weighted avg       0.51      0.52      0.51       207



In [ ]:
# ============================================================
# STEP 8: Predict New / Unseen Customers
# ============================================================
# This is the real business value — classifying new customers
# who walk into the store or sign up online.

print("=" * 60)
print("PREDICTING NEW SENSOR SAMPLES")
print("=" * 60)

# Hypothetical air samples for testing the trained model
new_samples = pd.DataFrame({
    'co_concentration': [1000, 1300, 1600],
    'o3_concentration': [700, 1100, 1500],
    'temperature':      [10.0, 15.0, 22.0]
})
# IMPORTANT: Scale new data using the SAME scaler fitted on training data
new_samples_scaled = scaler.transform(new_samples)
# Get predictions and probabilities
new_pred = model.predict(new_samples_scaled)
new_proba = model.predict_proba(new_samples_scaled)

for i in range(len(new_samples)):
    print(f"\n  Sample {i+1}:")
    print(f"    CO Sensor: {new_samples.iloc[i]['co_concentration']}, "
          f"O3 Sensor: {new_samples.iloc[i]['o3_concentration']}, "
          f"Temp: {new_samples.iloc[i]['temperature']}°C")
    print(f"    Predicted Tier: {le.classes_[new_pred[i]]}")
    print(f"    Probabilities:")
    for j, class_name in enumerate(le.classes_):
        bar = "█" * int(new_proba[i][j] * 30)
        print(f"      {class_name:>8s}: {new_proba[i][j]:.3f} {bar}")


PREDICTING NEW SENSOR SAMPLES

  Sample 1:
    CO Sensor: 1000.0, O3 Sensor: 700.0, Temp: 10.0°C
    Predicted Tier: moderate
    Probabilities:
          good: 0.379 ███████████
      moderate: 0.397 ███████████
          poor: 0.224 ██████

  Sample 2:
    CO Sensor: 1300.0, O3 Sensor: 1100.0, Temp: 15.0°C
    Predicted Tier: good
    Probabilities:
          good: 0.410 ████████████
      moderate: 0.337 ██████████
          poor: 0.253 ███████

  Sample 3:
    CO Sensor: 1600.0, O3 Sensor: 1500.0, Temp: 22.0°C
    Predicted Tier: good
    Probabilities:
          good: 0.361 ██████████
      moderate: 0.296 ████████
          poor: 0.343 ██████████


In [ ]:
# ============================================================
# STEP 9: Model Coefficients Interpretation
# ============================================================
# Each class has its own coefficients. The sign and magnitude
# tell us how each feature influences classification.

print("\n" + "=" * 60)
print("MODEL COEFFICIENTS (Feature Importance)")
print("=" * 60)

coef_df = pd.DataFrame(
    model.coef_,                                # shape: (3 classes, 3 features)
    columns=X.columns,                          # feature names
    index=le.classes_                           # class names
)
print(coef_df.round(4))

print(f"\nInterpretation:")
print(f"  - Positive coefficient → feature INCREASES probability of that class")
print(f"  - Negative coefficient → feature DECREASES probability of that class")
print(f"  - Larger absolute value → stronger influence")

# Show the intercepts too
print(f"\nModel Intercepts (bias terms):")
for i, name in enumerate(le.classes_):
    print(f"  {name}: {model.intercept_[i]:.4f}")

print("\n" + "=" * 60)
print("DONE! Air Quality Analysis Complete.")
print("=" * 60)



MODEL COEFFICIENTS (Feature Importance)
          co_concentration  o3_concentration  temperature
good                0.6133           -0.1726      -0.5037
moderate            0.1456           -0.4361       0.0749
poor               -0.7590            0.6087       0.4287

Interpretation:
  - Positive coefficient → feature INCREASES probability of that class
  - Negative coefficient → feature DECREASES probability of that class
  - Larger absolute value → stronger influence

Model Intercepts (bias terms):
  good: -0.0468
  moderate: 0.0441
  poor: 0.0027

DONE! Air Quality Analysis Complete.


# Summary
This program uses air sensor readings to sort air conditions into three categories: good, moderate, or poor. It cleans the dataset, keeps three key measurements (CO sensor, O3 sensor, and temperature), then trains a model that learns patterns that separate the three categories. It checks performance using repeated testing methods to make sure results are consistent, then reports how often it’s correct and where it gets confused between categories. It can also take new sensor readings and output both a predicted category and a confidence score—however, the current “good/moderate/poor” labels are created by splitting the dataset into thirds rather than using real air-quality rules, so it’s best viewed as a working demo unless the labels are updated to follow true air quality standards.